# Step 4D — Dataset-Wide Ego-Oriented Hierarchical BST Retrieval

This notebook runs the full **ego-oriented BST-BEV retrieval experiment** over all nuScenes-mini keyframes.

It compares, at **z23 and z24**:

- fixed-resolution target-zoom polygon coverage;
- optimized mixed-resolution hierarchical polygon coverage;
- candidate-set equivalence;
- retrieval precision / recall against exact oriented-polygon ground truth;
- descriptor compression;
- spatial over-coverage;
- cover-generation latency;
- map-index lookup latency;
- total online latency;
- speedup;
- performance as a function of ego orientation.

The hierarchical HD-map index is built **once per map location** and reused for every frame in that map.

## Required project files

This notebook assumes these already exist:

- `src/bst_bev/query/oriented_region.py`
- `src/bst_bev/query/polygon_hierarchical_fast.py`
- `src/bst_bev/nuscenes/pose.py`
- `src/bst_bev/maps/hierarchical_bst_map_index.py`

Expected outputs:

- `outputs/step4d_oriented_all_samples.csv`
- `outputs/step4d_oriented_summary.csv`
- `outputs/step4d_oriented_storage.csv`
- `outputs/step4d_orientation_analysis.csv`


In [1]:
from pathlib import Path
import sys
import math
import time
import gc

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DATAROOT = PROJECT_ROOT / "data" / "nuscenes"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Project root:", PROJECT_ROOT)
print("nuScenes root:", DATAROOT)
print("Output dir:", OUTPUT_DIR)


Project root: /Users/abinavbala/College/Research/BST-BEV
nuScenes root: /Users/abinavbala/College/Research/BST-BEV/data/nuscenes
Output dir: /Users/abinavbala/College/Research/BST-BEV/outputs


In [2]:
from nuscenes.nuscenes import NuScenes
from nuscenes.map_expansion.map_api import NuScenesMap

from shapely.geometry import box

from bst_id.region_algebra import Cell

from bst_bev.nuscenes.pose import quaternion_to_yaw

from bst_bev.query.oriented_region import (
    OrientedBEVRegion,
)

from bst_bev.query.polygon_hierarchical_fast import (
    FastHierarchicalPolygonRegionCover,
)

from bst_bev.maps.hierarchical_bst_map_index import (
    HierarchicalBSTMapIndex,
)


## Configuration

The BEV footprint remains **60 m × 60 m**, matching the earlier Step 3D baseline.

`MIN_ZOOM = 19` is the optimized lower hierarchy bound that supports the query cells encountered by the current workload.


In [3]:
MAP_LAYERS = (
    "lane_divider",
    "road_divider",
    "ped_crossing",
)

ZOOMS = [
    23,
    24,
]

MIN_ZOOM = 19
MAX_ZOOM = 24

BEV_FRONT_M = 30.0
BEV_BACK_M = 30.0
BEV_LEFT_M = 30.0
BEV_RIGHT_M = 30.0

INDEX_SPACING_M = 1.0
BOUNDARY_STEP_M = 1.0

# Repeated timing inside each frame.
# Five repeats is a reasonable notebook-level compromise.
TIMING_REPEATS = 5

RAW_OUTPUT = (
    OUTPUT_DIR
    / "step4d_oriented_all_samples.csv"
)

SUMMARY_OUTPUT = (
    OUTPUT_DIR
    / "step4d_oriented_summary.csv"
)

STORAGE_OUTPUT = (
    OUTPUT_DIR
    / "step4d_oriented_storage.csv"
)

ORIENTATION_OUTPUT = (
    OUTPUT_DIR
    / "step4d_orientation_analysis.csv"
)


## Load nuScenes-mini


In [4]:
nusc = NuScenes(
    version="v1.0-mini",
    dataroot=str(DATAROOT),
    verbose=False,
)

print("Scenes:", len(nusc.scene))
print("Samples:", len(nusc.sample))


Scenes: 10
Samples: 404


## Build sample metadata with ego yaw


In [5]:
def get_sample_context(
    nusc,
    sample,
):
    scene = nusc.get(
        "scene",
        sample["scene_token"],
    )

    log = nusc.get(
        "log",
        scene["log_token"],
    )

    location = log["location"]

    lidar = nusc.get(
        "sample_data",
        sample["data"]["LIDAR_TOP"],
    )

    ego_pose = nusc.get(
        "ego_pose",
        lidar["ego_pose_token"],
    )

    ego_x, ego_y, _ = (
        ego_pose["translation"]
    )

    ego_yaw = quaternion_to_yaw(
        ego_pose["rotation"]
    )

    return {
        "sample_token":
            sample["token"],

        "scene":
            scene["name"],

        "timestamp":
            sample["timestamp"],

        "location":
            location,

        "ego_x":
            float(ego_x),

        "ego_y":
            float(ego_y),

        "ego_yaw":
            float(ego_yaw),
    }


samples_df = pd.DataFrame(
    get_sample_context(
        nusc,
        sample,
    )
    for sample in nusc.sample
)

print("Rows:", len(samples_df))

samples_df.head()


Rows: 404


,sample_token,scene,timestamp,location,ego_x,ego_y,ego_yaw
0,ca9a282c9e77460f8360f564131a8af5,scene-0061,1532402927647951,singapore-onenorth,411.303935,1180.890379,-1.923645
1,39586f9d59004284a7114a68825e8eec,scene-0061,1532402928147847,singapore-onenorth,409.743152,1176.676973,-1.917831
2,356d81f38dd9473ba590f39e266f54e5,scene-0061,1532402928698048,singapore-onenorth,408.101879,1172.151578,-1.911575
3,e0845f5322254dafadbbed75aaa07969,scene-0061,1532402929197353,singapore-onenorth,406.726006,1168.211287,-1.900030
4,c923fe08b2ff4e27975d2bf30934383b,scene-0061,1532402929697797,singapore-onenorth,405.409406,1164.354227,-1.895311


In [6]:
assert len(samples_df) == 404

print(
    samples_df
    .groupby("location")
    .size()
)


location
boston-seaport              163
singapore-hollandvillage    121
singapore-onenorth           39
singapore-queenstown         81
dtype: int64


## Load each nuScenes HD map once


In [7]:
locations = sorted(
    samples_df["location"].unique()
)

nusc_maps = {
    location: NuScenesMap(
        dataroot=str(DATAROOT),
        map_name=location,
    )
    for location in locations
}

print("Loaded maps:")
for location in locations:
    print(" -", location)


Loaded maps:
 - boston-seaport
 - singapore-hollandvillage
 - singapore-onenorth
 - singapore-queenstown


## Exact oriented-BEV ground truth

Ground truth is computed in two stages:

1. use the oriented polygon's axis-aligned bounding box to cheaply retrieve nearby map records;
2. perform exact Shapely intersection against the oriented polygon.

This preprocessing is **not included in online query timing**.


In [8]:
def get_map_record_geometry(
    nusc_map,
    layer,
    record,
):
    if layer in (
        "lane_divider",
        "road_divider",
    ):
        return nusc_map.extract_line(
            record["line_token"]
        )

    if layer == "ped_crossing":
        return nusc_map.extract_polygon(
            record["polygon_token"]
        )

    raise ValueError(
        f"Unsupported layer: {layer}"
    )


def get_oriented_ground_truth_tokens(
    nusc_map,
    query_polygon,
    map_layers=MAP_LAYERS,
):
    min_x, min_y, max_x, max_y = (
        query_polygon.bounds
    )

    patch_candidates = (
        nusc_map.get_records_in_patch(
            (
                min_x,
                min_y,
                max_x,
                max_y,
            ),
            list(map_layers),
            mode="intersect",
        )
    )

    hits = set()

    for layer, tokens in (
        patch_candidates.items()
    ):
        for token in tokens:
            record = nusc_map.get(
                layer,
                token,
            )

            geometry = get_map_record_geometry(
                nusc_map,
                layer,
                record,
            )

            if geometry.intersects(
                query_polygon
            ):
                hits.add(
                    (
                        layer,
                        token,
                    )
                )

    return hits


In [9]:
oriented_gt_cache = {}

gt_start = time.perf_counter()

for _, row in samples_df.iterrows():

    location = str(
        row["location"]
    )

    bev = OrientedBEVRegion(
        center_x=float(
            row["ego_x"]
        ),
        center_y=float(
            row["ego_y"]
        ),
        yaw=float(
            row["ego_yaw"]
        ),
        front=BEV_FRONT_M,
        back=BEV_BACK_M,
        left=BEV_LEFT_M,
        right=BEV_RIGHT_M,
    )

    oriented_gt_cache[
        row["sample_token"]
    ] = (
        get_oriented_ground_truth_tokens(
            nusc_map=nusc_maps[
                location
            ],
            query_polygon=bev.polygon,
        )
    )

gt_time_s = (
    time.perf_counter()
    - gt_start
)

print(
    "Cached GT for",
    len(oriented_gt_cache),
    "samples",
)

print(
    "GT preprocessing:",
    f"{gt_time_s:.2f} s",
)


Cached GT for 404 samples
GT preprocessing: 5.30 s


## Fixed-resolution oriented polygon cover

This enumerates every target-zoom BST leaf cell intersecting the same projected oriented polygon.

It is the fixed-resolution baseline for Step 4D.


In [10]:
def fixed_leaf_polygon_cover(
    tile_polygon,
    target_zoom,
):
    min_x, min_y, max_x, max_y = (
        tile_polygon.bounds
    )

    x_start = math.floor(
        min_x
    )
    x_end = math.floor(
        max_x
    )

    y_start = math.floor(
        min_y
    )
    y_end = math.floor(
        max_y
    )

    cells = []

    for tile_x in range(
        x_start,
        x_end + 1,
    ):
        for tile_y in range(
            y_start,
            y_end + 1,
        ):

            leaf_polygon = box(
                tile_x,
                tile_y,
                tile_x + 1,
                tile_y + 1,
            )

            if leaf_polygon.disjoint(
                tile_polygon
            ):
                continue

            cells.append(
                Cell.from_parts(
                    x=(
                        target_zoom,
                        tile_x,
                    ),
                    y=(
                        target_zoom,
                        tile_y,
                    ),
                )
            )

    return tuple(
        cells
    )


## Retrieval and timing helpers


In [11]:
def retrieval_metrics(
    candidates,
    ground_truth,
):
    tp = len(
        candidates
        & ground_truth
    )

    fp = len(
        candidates
        - ground_truth
    )

    fn = len(
        ground_truth
        - candidates
    )

    precision = (
        tp / (tp + fp)
        if tp + fp
        else 0.0
    )

    recall = (
        tp / (tp + fn)
        if tp + fn
        else 0.0
    )

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
    }


def median_runtime_ms(
    fn,
    repeats=TIMING_REPEATS,
):
    timings = []

    for _ in range(
        repeats
    ):
        start = (
            time.perf_counter()
        )

        fn()

        timings.append(
            (
                time.perf_counter()
                - start
            )
            * 1000.0
        )

    return float(
        np.median(
            timings
        )
    )


## Evaluate one oriented frame


In [12]:
def evaluate_oriented_sample(
    *,
    row,
    gt_tokens,
    hierarchical_index,
    hierarchical_coverer,
    target_zoom,
):
    # ========================================================
    # BEV
    # ========================================================

    bev = OrientedBEVRegion(
        center_x=float(
            row["ego_x"]
        ),
        center_y=float(
            row["ego_y"]
        ),
        yaw=float(
            row["ego_yaw"]
        ),
        front=BEV_FRONT_M,
        back=BEV_BACK_M,
        left=BEV_LEFT_M,
        right=BEV_RIGHT_M,
    )

    # ========================================================
    # Covers
    # ========================================================

    hierarchical_cover = (
        hierarchical_coverer.cover(
            bev
        )
    )

    fixed_cells = (
        fixed_leaf_polygon_cover(
            tile_polygon=(
                hierarchical_cover
                .tile_polygon
            ),
            target_zoom=(
                target_zoom
            ),
        )
    )

    # ========================================================
    # Retrieval
    # ========================================================

    fixed_result = (
        hierarchical_index.query_cells(
            fixed_cells
        )
    )

    hierarchical_result = (
        hierarchical_index.query_cells(
            hierarchical_cover.cells
        )
    )

    fixed_tokens = (
        fixed_result.tokens
    )

    hierarchical_tokens = (
        hierarchical_result.tokens
    )

    # ========================================================
    # Candidate equivalence
    # ========================================================

    candidate_union = (
        fixed_tokens
        | hierarchical_tokens
    )

    candidate_intersection = (
        fixed_tokens
        & hierarchical_tokens
    )

    candidate_jaccard = (
        len(candidate_intersection)
        / len(candidate_union)
        if candidate_union
        else 1.0
    )

    same_candidates = (
        fixed_tokens
        == hierarchical_tokens
    )

    # ========================================================
    # Retrieval metrics
    # ========================================================

    fixed_metrics = (
        retrieval_metrics(
            fixed_tokens,
            gt_tokens,
        )
    )

    hierarchical_metrics = (
        retrieval_metrics(
            hierarchical_tokens,
            gt_tokens,
        )
    )

    # ========================================================
    # Representation metrics
    # ========================================================

    fixed_cell_count = len(
        fixed_cells
    )

    hierarchical_cell_count = (
        hierarchical_cover
        .hierarchical_cell_count
    )

    compression_ratio = (
        fixed_cell_count
        / hierarchical_cell_count
        if hierarchical_cell_count
        else np.nan
    )

    reduction_fraction = (
        1.0
        - hierarchical_cell_count
        / fixed_cell_count
        if fixed_cell_count
        else 0.0
    )

    polygon_area = (
        hierarchical_cover
        .tile_polygon
        .area
    )

    # A target-zoom leaf occupies one unit of tile-space area.
    fixed_cover_area = float(
        fixed_cell_count
    )

    # Hierarchical cells are non-overlapping.
    hierarchical_cover_area = float(
        hierarchical_cover
        .leaf_equivalent_count
    )

    fixed_overcoverage_ratio = (
        (
            fixed_cover_area
            - polygon_area
        )
        / polygon_area
    )

    hierarchical_overcoverage_ratio = (
        (
            hierarchical_cover_area
            - polygon_area
        )
        / polygon_area
    )

    # ========================================================
    # Stable timings
    # ========================================================

    tile_polygon = (
        hierarchical_cover
        .tile_polygon
    )

    fixed_cover_time_ms = (
        median_runtime_ms(
            lambda:
            fixed_leaf_polygon_cover(
                tile_polygon=(
                    tile_polygon
                ),
                target_zoom=(
                    target_zoom
                ),
            )
        )
    )

    hierarchical_cover_time_ms = (
        median_runtime_ms(
            lambda:
            hierarchical_coverer.cover(
                bev
            )
        )
    )

    fixed_lookup_time_ms = (
        median_runtime_ms(
            lambda:
            hierarchical_index.query_cells(
                fixed_cells
            )
        )
    )

    hierarchical_lookup_time_ms = (
        median_runtime_ms(
            lambda:
            hierarchical_index.query_cells(
                hierarchical_cover.cells
            )
        )
    )

    fixed_total_time_ms = (
        fixed_cover_time_ms
        + fixed_lookup_time_ms
    )

    hierarchical_total_time_ms = (
        hierarchical_cover_time_ms
        + hierarchical_lookup_time_ms
    )

    lookup_speedup = (
        fixed_lookup_time_ms
        / hierarchical_lookup_time_ms
        if hierarchical_lookup_time_ms > 0
        else np.nan
    )

    total_speedup = (
        fixed_total_time_ms
        / hierarchical_total_time_ms
        if hierarchical_total_time_ms > 0
        else np.nan
    )

    return {
        # Representation
        "fixed_cells":
            fixed_cell_count,

        "hierarchical_cells":
            hierarchical_cell_count,

        "compression_ratio":
            compression_ratio,

        "reduction_fraction":
            reduction_fraction,

        "polygon_area_tile":
            polygon_area,

        "fixed_overcoverage_ratio":
            fixed_overcoverage_ratio,

        "hierarchical_overcoverage_ratio":
            hierarchical_overcoverage_ratio,

        # Traversal
        "visited_cells":
            hierarchical_cover
            .visited_cell_count,

        "interior_cells":
            hierarchical_cover
            .interior_cell_count,

        "boundary_cells":
            hierarchical_cover
            .boundary_cell_count,

        "discarded_cells":
            hierarchical_cover
            .discarded_cell_count,

        # Candidates
        "fixed_candidates":
            len(fixed_tokens),

        "hierarchical_candidates":
            len(hierarchical_tokens),

        "same_candidates":
            same_candidates,

        "candidate_jaccard":
            candidate_jaccard,

        "fixed_only_count":
            len(
                fixed_tokens
                - hierarchical_tokens
            ),

        "hierarchical_only_count":
            len(
                hierarchical_tokens
                - fixed_tokens
            ),

        # Ground truth
        "ground_truth":
            len(gt_tokens),

        # Fixed retrieval
        "fixed_tp":
            fixed_metrics["tp"],

        "fixed_fp":
            fixed_metrics["fp"],

        "fixed_fn":
            fixed_metrics["fn"],

        "fixed_precision":
            fixed_metrics[
                "precision"
            ],

        "fixed_recall":
            fixed_metrics[
                "recall"
            ],

        # Hierarchical retrieval
        "hierarchical_tp":
            hierarchical_metrics[
                "tp"
            ],

        "hierarchical_fp":
            hierarchical_metrics[
                "fp"
            ],

        "hierarchical_fn":
            hierarchical_metrics[
                "fn"
            ],

        "hierarchical_precision":
            hierarchical_metrics[
                "precision"
            ],

        "hierarchical_recall":
            hierarchical_metrics[
                "recall"
            ],

        # Timing
        "fixed_cover_time_ms":
            fixed_cover_time_ms,

        "hierarchical_cover_time_ms":
            hierarchical_cover_time_ms,

        "fixed_lookup_time_ms":
            fixed_lookup_time_ms,

        "hierarchical_lookup_time_ms":
            hierarchical_lookup_time_ms,

        "fixed_total_time_ms":
            fixed_total_time_ms,

        "hierarchical_total_time_ms":
            hierarchical_total_time_ms,

        "lookup_speedup":
            lookup_speedup,

        "total_speedup":
            total_speedup,
    }


## Run Step 4D over all 404 samples × z23/z24

The hierarchical HD-map index is built once for each map location.

Intermediate CSV checkpoints are written after every `(location, zoom)` block.


In [13]:
all_results = []
storage_results = []

experiment_start = (
    time.perf_counter()
)


for location in locations:

    print()
    print("=" * 72)
    print(location)
    print("=" * 72)

    nusc_map = (
        nusc_maps[
            location
        ]
    )

    location_samples = (
        samples_df[
            samples_df["location"]
            == location
        ]
        .reset_index(
            drop=True
        )
    )

    # ========================================================
    # Hierarchical map index — build once per location
    # ========================================================

    print(
        "Building hierarchical map index..."
    )

    build_start = (
        time.perf_counter()
    )

    hierarchical_index = (
        HierarchicalBSTMapIndex(
            nusc_map=nusc_map,
            location=location,
            min_zoom=MIN_ZOOM,
            max_zoom=MAX_ZOOM,
            index_spacing_m=(
                INDEX_SPACING_M
            ),
        )
    )

    hierarchical_index.build()

    build_time_s = (
        time.perf_counter()
        - build_start
    )

    print(
        "  keys:",
        f"{hierarchical_index.key_count:,}",
    )

    print(
        "  memberships:",
        f"{hierarchical_index.membership_count:,}",
    )

    print(
        "  build:",
        f"{build_time_s:.2f} s",
    )

    storage_results.append({
        "location":
            location,

        "hierarchical_keys":
            hierarchical_index.key_count,

        "hierarchical_memberships":
            hierarchical_index.membership_count,

        "hierarchical_build_time_s":
            build_time_s,
    })

    # ========================================================
    # Target zooms
    # ========================================================

    for zoom in ZOOMS:

        print()
        print(
            f"  Zoom {zoom}"
        )

        hierarchical_coverer = (
            FastHierarchicalPolygonRegionCover(
                encoder=(
                    hierarchical_index
                    .encoder
                ),
                min_zoom=MIN_ZOOM,
                target_zoom=zoom,
                boundary_step_m=(
                    BOUNDARY_STEP_M
                ),
            )
        )

        # ----------------------------------------------------
        # Warm-up
        # ----------------------------------------------------

        warmup_row = (
            location_samples.iloc[0]
        )

        warmup_bev = (
            OrientedBEVRegion(
                center_x=float(
                    warmup_row["ego_x"]
                ),
                center_y=float(
                    warmup_row["ego_y"]
                ),
                yaw=float(
                    warmup_row["ego_yaw"]
                ),
                front=BEV_FRONT_M,
                back=BEV_BACK_M,
                left=BEV_LEFT_M,
                right=BEV_RIGHT_M,
            )
        )

        warmup_cover = (
            hierarchical_coverer.cover(
                warmup_bev
            )
        )

        warmup_fixed = (
            fixed_leaf_polygon_cover(
                tile_polygon=(
                    warmup_cover
                    .tile_polygon
                ),
                target_zoom=zoom,
            )
        )

        hierarchical_index.query_cells(
            warmup_cover.cells
        )

        hierarchical_index.query_cells(
            warmup_fixed
        )

        # ----------------------------------------------------
        # All frames for this location
        # ----------------------------------------------------

        for sample_number, row in (
            location_samples.iterrows()
        ):

            gt_tokens = (
                oriented_gt_cache[
                    row["sample_token"]
                ]
            )

            metrics = (
                evaluate_oriented_sample(
                    row=row,
                    gt_tokens=gt_tokens,
                    hierarchical_index=(
                        hierarchical_index
                    ),
                    hierarchical_coverer=(
                        hierarchical_coverer
                    ),
                    target_zoom=zoom,
                )
            )

            all_results.append({
                "zoom":
                    zoom,

                "scene":
                    row["scene"],

                "sample_token":
                    row["sample_token"],

                "timestamp":
                    row["timestamp"],

                "location":
                    location,

                "ego_x":
                    row["ego_x"],

                "ego_y":
                    row["ego_y"],

                "ego_yaw_rad":
                    row["ego_yaw"],

                "ego_yaw_deg":
                    math.degrees(
                        row["ego_yaw"]
                    ),

                **metrics,
            })

            if (
                (sample_number + 1)
                % 50
                == 0
            ):
                print(
                    f"    "
                    f"{sample_number + 1}/"
                    f"{len(location_samples)}"
                )

        # ----------------------------------------------------
        # Checkpoint
        # ----------------------------------------------------

        results_df = pd.DataFrame(
            all_results
        )

        results_df.to_csv(
            RAW_OUTPUT,
            index=False,
        )

        storage_df = pd.DataFrame(
            storage_results
        )

        storage_df.to_csv(
            STORAGE_OUTPUT,
            index=False,
        )

        block = (
            results_df[
                (
                    results_df[
                        "zoom"
                    ]
                    == zoom
                )
                &
                (
                    results_df[
                        "location"
                    ]
                    == location
                )
            ]
        )

        print(
            "    samples:",
            len(block),
        )

        print(
            "    agreement:",
            f"{block['same_candidates'].mean():.3f}",
        )

        print(
            "    median compression:",
            f"{block['compression_ratio'].median():.2f}x",
        )

        print(
            "    median speedup:",
            f"{block['total_speedup'].median():.2f}x",
        )

    del hierarchical_index
    gc.collect()


experiment_time_s = (
    time.perf_counter()
    - experiment_start
)

results_df = pd.DataFrame(
    all_results
)

storage_df = pd.DataFrame(
    storage_results
)

print()
print(
    "Experiment complete in",
    f"{experiment_time_s / 60:.2f}",
    "minutes",
)

print(
    "Rows:",
    len(results_df),
)



boston-seaport
Building hierarchical map index...
  keys: 297,437
  memberships: 466,330
  build: 2.20 s

  Zoom 23
    50/163
    100/163
    150/163
    samples: 163
    agreement: 1.000
    median compression: 2.41x
    median speedup: 2.40x

  Zoom 24
    50/163
    100/163
    150/163
    samples: 163
    agreement: 1.000
    median compression: 4.24x
    median speedup: 4.64x

singapore-hollandvillage
Building hierarchical map index...
  keys: 70,414
  memberships: 100,162
  build: 0.32 s

  Zoom 23
    50/121
    100/121
    samples: 121
    agreement: 1.000
    median compression: 1.95x
    median speedup: 2.33x

  Zoom 24
    50/121
    100/121
    samples: 121
    agreement: 1.000
    median compression: 3.29x
    median speedup: 4.47x

singapore-onenorth
Building hierarchical map index...
  keys: 103,158
  memberships: 148,255
  build: 0.71 s

  Zoom 23
    samples: 39
    agreement: 1.000
    median compression: 2.01x
    median speedup: 2.14x

  Zoom 24
    samples: 39
  

In [14]:
assert len(results_df) == (
    len(samples_df)
    * len(ZOOMS)
)

print(
    "Expected rows:",
    len(samples_df)
    * len(ZOOMS),
)

print(
    "Actual rows:",
    len(results_df),
)


Expected rows: 808
Actual rows: 808


## Correctness check

Do this before interpreting performance.


In [15]:
agreement = (
    results_df
    .groupby("zoom")
    .agg(
        samples=(
            "sample_token",
            "count",
        ),

        exact_matches=(
            "same_candidates",
            "sum",
        ),

        agreement_rate=(
            "same_candidates",
            "mean",
        ),

        minimum_jaccard=(
            "candidate_jaccard",
            "min",
        ),

        mean_jaccard=(
            "candidate_jaccard",
            "mean",
        ),

        fixed_only=(
            "fixed_only_count",
            "sum",
        ),

        hierarchical_only=(
            "hierarchical_only_count",
            "sum",
        ),
    )
)

agreement


,samples,exact_matches,agreement_rate,minimum_jaccard,mean_jaccard,fixed_only,hierarchical_only
zoom,,,,,,,
23,404,404,1.0,1.0,1.0,0,0
24,404,404,1.0,1.0,1.0,0,0


In [16]:
mismatches = (
    results_df[
        ~results_df[
            "same_candidates"
        ]
    ]
)

print(
    "Candidate mismatches:",
    len(mismatches),
)

mismatches[
    [
        "zoom",
        "scene",
        "location",
        "ego_yaw_deg",
        "fixed_cells",
        "hierarchical_cells",
        "fixed_candidates",
        "hierarchical_candidates",
        "candidate_jaccard",
        "fixed_only_count",
        "hierarchical_only_count",
    ]
].head(30)


Candidate mismatches: 0


,zoom,scene,location,ego_yaw_deg,fixed_cells,hierarchical_cells,fixed_candidates,hierarchical_candidates,candidate_jaccard,fixed_only_count,hierarchical_only_count


## Dataset-wide summary


In [17]:
summary = (
    results_df
    .groupby("zoom")
    .agg(
        samples=(
            "sample_token",
            "count",
        ),

        # Candidate equivalence
        exact_candidate_matches=(
            "same_candidates",
            "sum",
        ),

        agreement_rate=(
            "same_candidates",
            "mean",
        ),

        min_candidate_jaccard=(
            "candidate_jaccard",
            "min",
        ),

        # Descriptor counts
        mean_fixed_cells=(
            "fixed_cells",
            "mean",
        ),

        median_fixed_cells=(
            "fixed_cells",
            "median",
        ),

        p95_fixed_cells=(
            "fixed_cells",
            lambda x:
                x.quantile(0.95),
        ),

        mean_hierarchical_cells=(
            "hierarchical_cells",
            "mean",
        ),

        median_hierarchical_cells=(
            "hierarchical_cells",
            "median",
        ),

        p95_hierarchical_cells=(
            "hierarchical_cells",
            lambda x:
                x.quantile(0.95),
        ),

        mean_compression_ratio=(
            "compression_ratio",
            "mean",
        ),

        median_compression_ratio=(
            "compression_ratio",
            "median",
        ),

        mean_reduction_fraction=(
            "reduction_fraction",
            "mean",
        ),

        # Over-coverage
        mean_fixed_overcoverage=(
            "fixed_overcoverage_ratio",
            "mean",
        ),

        median_fixed_overcoverage=(
            "fixed_overcoverage_ratio",
            "median",
        ),

        mean_hierarchical_overcoverage=(
            "hierarchical_overcoverage_ratio",
            "mean",
        ),

        median_hierarchical_overcoverage=(
            "hierarchical_overcoverage_ratio",
            "median",
        ),

        p95_hierarchical_overcoverage=(
            "hierarchical_overcoverage_ratio",
            lambda x:
                x.quantile(0.95),
        ),

        # Retrieval counts
        fixed_tp=(
            "fixed_tp",
            "sum",
        ),

        fixed_fp=(
            "fixed_fp",
            "sum",
        ),

        fixed_fn=(
            "fixed_fn",
            "sum",
        ),

        hierarchical_tp=(
            "hierarchical_tp",
            "sum",
        ),

        hierarchical_fp=(
            "hierarchical_fp",
            "sum",
        ),

        hierarchical_fn=(
            "hierarchical_fn",
            "sum",
        ),

        # Cover latency
        median_fixed_cover_ms=(
            "fixed_cover_time_ms",
            "median",
        ),

        p95_fixed_cover_ms=(
            "fixed_cover_time_ms",
            lambda x:
                x.quantile(0.95),
        ),

        median_hierarchical_cover_ms=(
            "hierarchical_cover_time_ms",
            "median",
        ),

        p95_hierarchical_cover_ms=(
            "hierarchical_cover_time_ms",
            lambda x:
                x.quantile(0.95),
        ),

        # Lookup latency
        median_fixed_lookup_ms=(
            "fixed_lookup_time_ms",
            "median",
        ),

        p95_fixed_lookup_ms=(
            "fixed_lookup_time_ms",
            lambda x:
                x.quantile(0.95),
        ),

        median_hierarchical_lookup_ms=(
            "hierarchical_lookup_time_ms",
            "median",
        ),

        p95_hierarchical_lookup_ms=(
            "hierarchical_lookup_time_ms",
            lambda x:
                x.quantile(0.95),
        ),

        # Total latency
        mean_fixed_total_ms=(
            "fixed_total_time_ms",
            "mean",
        ),

        median_fixed_total_ms=(
            "fixed_total_time_ms",
            "median",
        ),

        p95_fixed_total_ms=(
            "fixed_total_time_ms",
            lambda x:
                x.quantile(0.95),
        ),

        mean_hierarchical_total_ms=(
            "hierarchical_total_time_ms",
            "mean",
        ),

        median_hierarchical_total_ms=(
            "hierarchical_total_time_ms",
            "median",
        ),

        p95_hierarchical_total_ms=(
            "hierarchical_total_time_ms",
            lambda x:
                x.quantile(0.95),
        ),

        # Speedup
        mean_total_speedup=(
            "total_speedup",
            "mean",
        ),

        median_total_speedup=(
            "total_speedup",
            "median",
        ),

        min_total_speedup=(
            "total_speedup",
            "min",
        ),

        p05_total_speedup=(
            "total_speedup",
            lambda x:
                x.quantile(0.05),
        ),

        median_lookup_speedup=(
            "lookup_speedup",
            "median",
        ),
    )
    .reset_index()
)


In [18]:
# Micro precision / recall.

summary[
    "fixed_precision"
] = (
    summary["fixed_tp"]
    /
    (
        summary["fixed_tp"]
        + summary["fixed_fp"]
    )
)

summary[
    "fixed_recall"
] = (
    summary["fixed_tp"]
    /
    (
        summary["fixed_tp"]
        + summary["fixed_fn"]
    )
)

summary[
    "hierarchical_precision"
] = (
    summary["hierarchical_tp"]
    /
    (
        summary["hierarchical_tp"]
        + summary["hierarchical_fp"]
    )
)

summary[
    "hierarchical_recall"
] = (
    summary["hierarchical_tp"]
    /
    (
        summary["hierarchical_tp"]
        + summary["hierarchical_fn"]
    )
)


In [19]:
final_summary = summary[
    [
        "zoom",
        "samples",

        "agreement_rate",
        "min_candidate_jaccard",

        "fixed_precision",
        "fixed_recall",

        "hierarchical_precision",
        "hierarchical_recall",

        "mean_fixed_cells",
        "mean_hierarchical_cells",

        "mean_compression_ratio",
        "median_compression_ratio",
        "mean_reduction_fraction",

        "mean_fixed_overcoverage",
        "mean_hierarchical_overcoverage",
        "p95_hierarchical_overcoverage",

        "median_fixed_cover_ms",
        "median_hierarchical_cover_ms",

        "median_fixed_lookup_ms",
        "median_hierarchical_lookup_ms",

        "median_fixed_total_ms",
        "median_hierarchical_total_ms",

        "p95_fixed_total_ms",
        "p95_hierarchical_total_ms",

        "median_lookup_speedup",
        "median_total_speedup",

        "p05_total_speedup",
        "min_total_speedup",
    ]
]

final_summary


,zoom,samples,agreement_rate,min_candidate_jaccard,fixed_precision,fixed_recall,hierarchical_precision,hierarchical_recall,mean_fixed_cells,mean_hierarchical_cells,...,median_fixed_lookup_ms,median_hierarchical_lookup_ms,median_fixed_total_ms,median_hierarchical_total_ms,p95_fixed_total_ms,p95_hierarchical_total_ms,median_lookup_speedup,median_total_speedup,p05_total_speedup,min_total_speedup
0,23,404,1.0,1.0,0.909353,1.0,0.909353,1.0,249.245050,112.529703,...,0.548208,0.288979,9.945354,4.114916,18.326565,7.340371,2.060690,2.351036,1.922159,1.640488
1,24,404,1.0,1.0,0.944175,1.0,0.944175,1.0,919.089109,241.289604,...,1.973666,0.617958,37.053417,8.051917,69.383933,14.437598,3.474813,4.543314,3.720851,3.568925


In [20]:
final_summary.to_csv(
    SUMMARY_OUTPUT,
    index=False,
)

storage_df.to_csv(
    STORAGE_OUTPUT,
    index=False,
)

results_df.to_csv(
    RAW_OUTPUT,
    index=False,
)

print("Saved:")
print(" -", RAW_OUTPUT)
print(" -", SUMMARY_OUTPUT)
print(" -", STORAGE_OUTPUT)


Saved:
 - /Users/abinavbala/College/Research/BST-BEV/outputs/step4d_oriented_all_samples.csv
 - /Users/abinavbala/College/Research/BST-BEV/outputs/step4d_oriented_summary.csv
 - /Users/abinavbala/College/Research/BST-BEV/outputs/step4d_oriented_storage.csv


## Human-readable compact table


In [21]:
display_summary = (
    final_summary.copy()
)

display_summary[
    "descriptor_reduction_pct"
] = (
    100
    * (
        1
        - display_summary[
            "mean_hierarchical_cells"
        ]
        / display_summary[
            "mean_fixed_cells"
        ]
    )
)

display_summary[
    "mean_hierarchical_overcoverage_pct"
] = (
    100
    * display_summary[
        "mean_hierarchical_overcoverage"
    ]
)

display_summary[
    "p95_hierarchical_overcoverage_pct"
] = (
    100
    * display_summary[
        "p95_hierarchical_overcoverage"
    ]
)

display_summary[
    [
        "zoom",
        "agreement_rate",

        "fixed_precision",
        "fixed_recall",

        "hierarchical_precision",
        "hierarchical_recall",

        "mean_fixed_cells",
        "mean_hierarchical_cells",

        "descriptor_reduction_pct",

        "mean_hierarchical_overcoverage_pct",

        "median_fixed_total_ms",
        "median_hierarchical_total_ms",

        "p95_fixed_total_ms",
        "p95_hierarchical_total_ms",

        "median_total_speedup",
        "p05_total_speedup",
        "min_total_speedup",
    ]
]


,zoom,agreement_rate,fixed_precision,fixed_recall,hierarchical_precision,hierarchical_recall,mean_fixed_cells,mean_hierarchical_cells,descriptor_reduction_pct,mean_hierarchical_overcoverage_pct,median_fixed_total_ms,median_hierarchical_total_ms,p95_fixed_total_ms,p95_hierarchical_total_ms,median_total_speedup,p05_total_speedup,min_total_speedup
0,23,1.0,0.909353,1.0,0.909353,1.0,249.245050,112.529703,54.851780,19.333478,9.945354,4.114916,18.326565,7.340371,2.351036,1.922159,1.640488
1,24,1.0,0.944175,1.0,0.944175,1.0,919.089109,241.289604,73.746876,9.536918,37.053417,8.051917,69.383933,14.437598,4.543314,3.720851,3.568925


## Orientation sensitivity

`axis_alignment_deg` measures the angular distance to the nearest global X/Y axis:

- `0°` = axis aligned;
- `45°` = maximally diagonal.

This checks whether compression and latency degrade as the BEV becomes less aligned with the BST grid.


In [22]:
def axis_alignment_angle(
    yaw_rad,
):
    angle = (
        abs(
            math.degrees(
                yaw_rad
            )
        )
        % 90.0
    )

    return min(
        angle,
        90.0 - angle,
    )


results_df[
    "axis_alignment_deg"
] = (
    results_df[
        "ego_yaw_rad"
    ].apply(
        axis_alignment_angle
    )
)

results_df[
    "orientation_bin"
] = pd.cut(
    results_df[
        "axis_alignment_deg"
    ],
    bins=[
        0,
        10,
        20,
        30,
        40,
        45.0001,
    ],
    include_lowest=True,
)


In [23]:
orientation_analysis = (
    results_df
    .groupby(
        [
            "zoom",
            "orientation_bin",
        ],
        observed=True,
    )
    .agg(
        samples=(
            "sample_token",
            "count",
        ),

        mean_fixed_cells=(
            "fixed_cells",
            "mean",
        ),

        mean_hierarchical_cells=(
            "hierarchical_cells",
            "mean",
        ),

        median_compression=(
            "compression_ratio",
            "median",
        ),

        median_overcoverage=(
            "hierarchical_overcoverage_ratio",
            "median",
        ),

        median_total_ms=(
            "hierarchical_total_time_ms",
            "median",
        ),

        median_speedup=(
            "total_speedup",
            "median",
        ),
    )
    .reset_index()
)

orientation_analysis


,zoom,orientation_bin,samples,mean_fixed_cells,mean_hierarchical_cells,median_compression,median_overcoverage,median_total_ms,median_speedup
0,23,"(-0.001, 10.0]",56,287.535714,101.089286,2.902260,0.125543,5.330292,1.956179
1,23,"(10.0, 20.0]",38,212.026316,97.763158,2.092869,0.197578,3.655375,2.104284
2,23,"(20.0, 30.0]",96,246.989583,112.458333,2.042553,0.216572,4.061521,2.308816
3,23,"(30.0, 40.0]",92,232.858696,113.619565,1.935044,0.226085,4.101000,2.411566
4,23,"(40.0, 45.0]",122,257.393443,121.614754,1.921949,0.210249,4.201791,2.487851
5,24,"(-0.001, 10.0]",56,1081.517857,214.214286,5.250087,0.061474,10.351458,3.763882
6,24,"(10.0, 20.0]",38,780.657895,208.947368,3.601036,0.099364,7.104833,4.115038
7,24,"(20.0, 30.0]",96,909.895833,241.697917,3.393204,0.105690,7.864250,4.434881
8,24,"(30.0, 40.0]",92,851.413043,243.760870,3.222477,0.112029,7.981666,4.641915
9,24,"(40.0, 45.0]",122,945.918033,261.606557,3.237882,0.108869,8.212312,4.776729


In [24]:
orientation_analysis.to_csv(
    ORIENTATION_OUTPUT,
    index=False,
)

print(
    "Saved:",
    ORIENTATION_OUTPUT,
)


Saved: /Users/abinavbala/College/Research/BST-BEV/outputs/step4d_orientation_analysis.csv


## Optional: inspect the slowest / weakest frames

These tables are useful if any frame behaves unexpectedly.


In [25]:
results_df[
    [
        "zoom",
        "scene",
        "location",
        "ego_yaw_deg",
        "axis_alignment_deg",
        "fixed_cells",
        "hierarchical_cells",
        "compression_ratio",
        "hierarchical_overcoverage_ratio",
        "fixed_total_time_ms",
        "hierarchical_total_time_ms",
        "total_speedup",
        "candidate_jaccard",
    ]
].sort_values(
    "total_speedup"
).head(20)


,zoom,scene,location,ego_yaw_deg,axis_alignment_deg,fixed_cells,hierarchical_cells,compression_ratio,hierarchical_overcoverage_ratio,fixed_total_time_ms,hierarchical_total_time_ms,total_speedup,candidate_jaccard
97,23,scene-0655,boston-seaport,0.626007,0.626007,324,90,3.600000,0.122080,10.313750,6.287001,1.640488,1.0
82,23,scene-0655,boston-seaport,1.395340,1.395340,324,129,2.511628,0.122081,10.169249,5.637583,1.803831,1.0
81,23,scene-0655,boston-seaport,1.324741,1.324741,324,119,2.722689,0.122081,9.940458,5.486501,1.811803,1.0
703,23,scene-0916,singapore-queenstown,171.729566,8.270434,189,90,2.100000,0.197582,6.661125,3.648833,1.825549,1.0
584,23,scene-0061,singapore-onenorth,-85.045542,4.954458,184,81,2.271605,0.165896,5.743791,3.138917,1.829864,1.0
688,23,scene-0916,singapore-queenstown,-94.577579,4.577579,185,72,2.569444,0.172236,5.657709,3.083833,1.834635,1.0
378,23,scene-1094,singapore-hollandvillage,178.795833,1.204167,181,75,2.413333,0.146876,5.651167,3.059876,1.846861,1.0
84,23,scene-0655,boston-seaport,1.479157,1.479157,324,129,2.511628,0.122080,10.933916,5.900959,1.852905,1.0
583,23,scene-0061,singapore-onenorth,-90.490642,0.490642,182,83,2.192771,0.153223,5.269625,2.827874,1.863458,1.0
582,23,scene-0061,singapore-onenorth,-95.599787,5.599787,184,87,2.114943,0.165896,6.012376,3.214708,1.870271,1.0


## Step 4D success criteria

A strong Step 4D result should show:

- **808 / 808** fixed-vs-hierarchical candidate-set agreement;
- minimum candidate Jaccard of **1.0**;
- hierarchical recall equal to fixed recall, ideally **1.0**;
- substantially fewer hierarchical descriptors;
- bounded conservative over-coverage;
- lower median and p95 online latency;
- median speedup greater than `1×` at both z23 and z24;
- preferably a `p05_total_speedup > 1`, indicating the hierarchical pipeline is faster on roughly 95% of frames.

This notebook is the ego-oriented hierarchical BST retrieval baseline to freeze before adding conventional spatial-index baselines or MapTR integration.
